# Enhanced Deltashare Commercial Reports Optimization

## Overview

This notebook compares two view definitions used to produce the enhanced commercial report for Vizio and ONN source data.

| View | Cell | Purpose |
| --- | --- | --- |
| `dev.mohit_gangwani.test_enhanced_kinetiq_2026070715_existing` | Cell 9 | Existing template-style definition |
| `dev.mohit_gangwani.test_enhanced_kinetiq_2026070715_refined` | Cell 10 | Refined definition with structural query changes |

The refined view keeps the same output shape and returned the same benchmark results as the existing view, but changes the query shape in three important ways:

* deduplicates `tv_populations` by `fk_tvid` before joining
* replaces `UNION` with `UNION ALL`
* precomputes `prev_filter` and `next_filter` so blacklist logic is evaluated once per row instead of once per projected column

The goal of the notebook is to show both the logical-plan differences and the measured runtime impact of those changes on serverless compute.

## Problem 1: Row Amplification from `tv_populations`

The existing view joins `tv_populations` to `populations` to find the `opted_in` audience, then joins that result directly to the commercial fact tables.

```sql
FROM prod.detection.tv_populations AS tp
JOIN prod.detection.populations AS pop
  ON tp.fk_population_id = pop.population_id
 AND LOWER(pop.population_name) = 'opted_in'
JOIN (...) AS mt
  ON mt.fk_tvid = tp.fk_tvid
```

The issue is that `tv_populations` contains multiple rows per `fk_tvid`. In the plan for Cell 12, the semi-joined `tv_populations` branch still carries about 1.05M rows for roughly 258K distinct devices. That means each commercial row can be duplicated several times before the final `DISTINCT` removes the fan-out.

Why this hurts performance:

* more rows flow into the main join than necessary
* the outer `SELECT DISTINCT` must remove duplicates created by the join
* the extra rows make the later shuffle and aggregate stages materially more expensive

## Fix 1: Deduplicate `tv_populations` Before the Join

The refined view collapses the opted-in device list to one row per `fk_tvid` before it is joined to the commercial rows:

```sql
FROM (
    SELECT tp.fk_tvid
    FROM prod.detection.tv_populations AS tp
    JOIN prod.detection.populations AS pop
      ON tp.fk_population_id = pop.population_id
     AND LOWER(pop.population_name) = 'opted_in'
    GROUP BY 1
) AS tp
JOIN (...) AS mt
  ON mt.fk_tvid = tp.fk_tvid
```

This changes the `tv_populations` branch from a many-row-per-device input into a distinct device set.

Observed effect in the optimized logical plan:

* `tv_populations` branch before dedup: about 1.05M rows
* after `GROUP BY fk_tvid`: about 258K rows
* reduction before the main join: about 75%

This is one of the highest-impact changes in the refined view because it removes row amplification before the expensive global `DISTINCT`.

## Problem 2: `UNION` and Repeated Per-Column Blacklist Logic

The existing view combines the Vizio and ONN branches with `UNION` and then repeats the same blacklist logic inside many projected columns.

```sql
SELECT viz.*
FROM prod.detection.viewing_commercials_cfe_golden AS viz
...
UNION
SELECT onn.*
FROM prod.detection_onn.viewing_commercials_cfe_golden AS onn
...
```

and later:

```sql
CASE WHEN mt.prev_nielsen_exclusive THEN NULL
     WHEN ...prev_station_blacklist_clients... THEN NULL
     ELSE ...
END
```

Why this hurts performance:

* `UNION` introduces a DISTINCT-style deduplication step across the two branches
* the same boolean logic is re-evaluated across many output columns
* larger intermediate rows and repeated expression evaluation increase the cost of the join, project, and aggregate stages

In practice, the optimizer still has to carry a more expensive plan shape into the downstream `DISTINCT`.

## Fix 2: `UNION ALL` Plus Precomputed Row Filters

The refined view makes two related changes inside the commercial-source subquery:

```sql
SELECT viz.*,
       viz.next_nielsen_exclusive
       OR (NOT(reported_input_source <=> 'ANTENNA') AND COALESCE(next_station_blacklist_clients, '||') = '||') AS next_filter,
       viz.prev_nielsen_exclusive
       OR (NOT(reported_input_source <=> 'ANTENNA') AND COALESCE(prev_station_blacklist_clients, '||') = '||') AS prev_filter
FROM prod.detection.viewing_commercials_cfe_golden AS viz
...
UNION ALL
SELECT onn.*,
       onn.next_nielsen_exclusive OR onn.next_station_blacklist_clients <=> '|ALL|'
       OR (...) AS next_filter,
       onn.prev_nielsen_exclusive OR onn.prev_station_blacklist_clients <=> '|ALL|'
       OR (...) AS prev_filter
FROM prod.detection_onn.viewing_commercials_cfe_golden AS onn
...
```

Benefits:

* `UNION ALL` removes the extra dedup step between the Vizio and ONN branches
* `prev_filter` and `next_filter` are computed once per row and reused throughout the outer SELECT
* the branch-specific business rules stay explicit: the Vizio side includes the `reported_input_source = 'ANTENNA'` condition, while the ONN side uses blacklist-list logic directly

The explain plan for Cell 13 confirms that the UNION node is now a direct `Union false, false` rather than an Aggregate-over-Union pattern.

## Problem 3: Repeated CASE Expressions Still Inflate the Existing Plan

In the existing view, the same blacklist and Nielsen checks are repeated across many output columns:

```sql
CASE WHEN mt.prev_nielsen_exclusive THEN NULL
     WHEN (mt.prev_station_blacklist_clients IS NOT NULL
           AND mt.prev_station_blacklist_clients != '||'
           AND mt.prev_station_blacklist_clients NOT LIKE '%|vizio|%')
           OR mt.prev_station_blacklist_clients <=> '|ALL|' THEN NULL
     ELSE ...
END
```

This pattern appears across the prev/next episode, title, callsign, network, and `live` columns.

Why this hurts performance:

* the expression tree is wider and repeated in many projected columns
* the optimizer cannot reuse a named boolean attribute in those CASE expressions
* more CPU time is spent evaluating the same logic repeatedly during projection

This is not the largest bottleneck in the notebook, but it contributes directly to the heavier Photon time observed in the existing query profile.

## Fix 3: Simplified Outer Projection

After `prev_filter` and `next_filter` are created in the source subquery, the outer projection becomes much simpler:

```sql
CASE WHEN mt.prev_filter THEN NULL
     ELSE mt.tms_prev_episode_id
END AS prev_episode_id,

CASE WHEN mt.next_filter THEN NULL
     ELSE COALESCE(mt.tms_next_show_title, mt.tivo_next_show_title)
END AS next_title
```

The explain plan for Cell 13 shows exactly this structure: the CASE expressions reference `prev_filter` and `next_filter` as simple attributes rather than expanding the blacklist logic in every column.

That simplification matters because it reduces per-row expression work across all projected columns while preserving the same downstream output.

## Expected Query Performance Impact

Based on the refined query shape, the expected wins were:

| Optimization | What it changes | Expected impact |
| --- | --- | --- |
| `GROUP BY fk_tvid` in `tv_populations` | Removes duplicate device rows before the join | Lower join fan-out and smaller downstream shuffle |
| `UNION ALL` instead of `UNION` | Removes branch-level dedup between Vizio and ONN | Lower aggregate overhead between source branches |
| `prev_filter` / `next_filter` columns | Reuses a single boolean in many CASE expressions | Lower projection CPU cost |

Those expectations were confirmed by the benchmark results later in the notebook:

* identical output from both views on the 3-day benchmark window
* materially lower task time, Photon CPU time, and shuffle time for the refined view
* lower remote bytes read despite both queries scanning similar row counts

The largest remaining cost is still the outer `SELECT DISTINCT`, which means the refined view is better than the existing one but not yet the end state for optimization.

In [0]:
%sql
CREATE OR REPLACE VIEW dev.mohit_gangwani.test_enhanced_kinetiq_2026070715_existing AS
SELECT /*+ BROADCAST(tp) 
      BROADCAST(pop)
      BROADCAST(v) 
         */ DISTINCT
    mt.tvid AS tvid, 
    substring(mt.zipcode, 1, 10) as zipcode, 
    substring(mt.dma, 1, 128) as dma, 
    substring(mt.external_id, 1, 256) as value, 
    mt.mt_start as mt_start, 
    mt.session_start as ts_start, 
    mt.session_end as ts_end, 
    substring((CASE WHEN 'vizio' != 'nielsen' AND mt.prev_nielsen_exclusive THEN NULL 
      WHEN (mt.prev_station_blacklist_clients IS NOT NULL 
            AND mt.prev_station_blacklist_clients != '||' 
            AND mt.prev_station_blacklist_clients NOT LIKE '%|vizio|%') OR mt.prev_station_blacklist_clients <=> '|ALL|' THEN NULL 
      ELSE CASE WHEN 'TMS' = 'TMS'  THEN mt.tms_prev_episode_id 
                WHEN 'TMS' = 'TIVO' THEN mt.tivo_prev_episode_id END 
 END ),1,256) as prev_episode_id, 
    substring((CASE WHEN 'vizio' != 'nielsen' AND mt.prev_nielsen_exclusive THEN NULL 
      WHEN (mt.prev_station_blacklist_clients IS NOT NULL 
            AND mt.prev_station_blacklist_clients != '||' 
            AND mt.prev_station_blacklist_clients NOT LIKE '%|vizio|%') OR mt.prev_station_blacklist_clients <=> '|ALL|' THEN NULL 
      WHEN 'vizio' = 'nielsen' THEN mt.tms_prev_show_title 
      ELSE CASE WHEN 'TMS' = 'TMS'  THEN COALESCE(mt.tms_prev_show_title, mt.tivo_prev_show_title) 
                WHEN 'TMS' = 'TIVO' THEN  COALESCE(mt.tivo_prev_show_title, mt.tms_prev_show_title) END 
 END ),1,256) as prev_title, 
    mt.prev_ts_start as prev_ts_start, 
    mt.prev_ts_end as prev_ts_end, 
    substring((CASE WHEN 'vizio' != 'nielsen' AND mt.prev_nielsen_exclusive THEN NULL 
      WHEN (mt.prev_station_blacklist_clients IS NOT NULL 
            AND mt.prev_station_blacklist_clients != '||' 
            AND mt.prev_station_blacklist_clients NOT LIKE '%|vizio|%') OR mt.prev_station_blacklist_clients <=> '|ALL|' THEN NULL 
      WHEN 'vizio' = 'nielsen' THEN mt.tms_prev_callsign 
      ELSE CASE WHEN 'TMS' = 'TMS'  THEN COALESCE(mt.tms_prev_callsign, mt.tivo_prev_callsign) 
                WHEN 'TMS' = 'TIVO' THEN COALESCE(mt.tivo_prev_callsign, mt.tms_prev_callsign) END 
 END ),1,30) as prev_channel_callsign, 
    substring((CASE WHEN 'vizio' != 'nielsen' AND mt.prev_nielsen_exclusive THEN NULL        
      WHEN (mt.prev_station_blacklist_clients IS NOT NULL 
            AND mt.prev_station_blacklist_clients != '||' 
            AND mt.prev_station_blacklist_clients NOT LIKE '%|vizio|%') OR mt.prev_station_blacklist_clients <=> '|ALL|' THEN NULL 
      WHEN 'vizio' = 'nielsen' THEN mt.tms_prev_network_affiliate 
      ELSE CASE WHEN 'TMS' = 'TMS'  THEN COALESCE(mt.tms_prev_network_affiliate, mt.tivo_prev_network_affiliate) 
                WHEN 'TMS' = 'TIVO' THEN COALESCE(mt.tivo_prev_network_affiliate, mt.tms_prev_network_affiliate) END 
 END),1,40) as prev_network_affiliate, 
    substring((CASE WHEN 'vizio' != 'nielsen' AND mt.next_nielsen_exclusive THEN NULL 
      WHEN (mt.next_station_blacklist_clients IS NOT NULL 
            AND mt.next_station_blacklist_clients != '||' 
            AND mt.next_station_blacklist_clients NOT LIKE '%|vizio|%') OR mt.next_station_blacklist_clients <=> '|ALL|' THEN NULL 
      ELSE CASE WHEN 'TMS' = 'TMS'  THEN mt.tms_next_episode_id 
                WHEN 'TMS' = 'TIVO' THEN mt.tivo_next_episode_id END 
 END),1,256) as next_episode_id, 
    substring((CASE WHEN 'vizio' != 'nielsen' AND mt.next_nielsen_exclusive THEN NULL 
      WHEN (mt.next_station_blacklist_clients IS NOT NULL 
           AND mt.next_station_blacklist_clients != '||' 
           AND mt.next_station_blacklist_clients NOT LIKE '%|vizio|%') OR mt.next_station_blacklist_clients <=> '|ALL|' THEN NULL 
      WHEN 'vizio' = 'nielsen' THEN mt.tms_next_show_title 
      ELSE CASE WHEN 'TMS' = 'TMS'  THEN COALESCE(mt.tms_next_show_title, mt.tivo_next_show_title) 
                WHEN 'TMS' = 'TIVO' THEN COALESCE(mt.tivo_next_show_title, mt.tms_next_show_title) END 
 END),1,256) as next_title, 
    mt.next_ts_start as next_ts_start, 
    mt.next_ts_end as next_ts_end, 
    substring((CASE WHEN 'vizio' != 'nielsen' AND mt.next_nielsen_exclusive THEN NULL 
      WHEN (mt.next_station_blacklist_clients IS NOT NULL 
            AND mt.next_station_blacklist_clients != '||' 
            AND mt.next_station_blacklist_clients NOT LIKE '%|vizio|%') OR mt.next_station_blacklist_clients <=> '|ALL|' THEN NULL 
      WHEN 'vizio' = 'nielsen' THEN mt.tms_next_callsign 
      ELSE CASE WHEN 'TMS' = 'TMS'  THEN COALESCE(mt.tms_next_callsign, mt.tivo_next_callsign) 
                WHEN 'TMS' = 'TIVO' THEN COALESCE(mt.tivo_next_callsign, mt.tms_next_callsign) END 
 END ),1,30) as next_channel_callsign, 
    substring((CASE WHEN 'vizio' != 'nielsen' AND mt.next_nielsen_exclusive THEN NULL 
      WHEN (mt.next_station_blacklist_clients IS NOT NULL 
            AND mt.next_station_blacklist_clients != '||' 
            AND mt.next_station_blacklist_clients NOT LIKE '%|vizio|%') OR mt.next_station_blacklist_clients <=> '|ALL|' THEN NULL 
      WHEN 'vizio' = 'nielsen' THEN mt.tms_next_network_affiliate 
      ELSE CASE WHEN 'TMS' = 'TMS'  THEN COALESCE(mt.tms_next_network_affiliate, mt.tivo_next_network_affiliate) 
                WHEN 'TMS' = 'TIVO' THEN COALESCE(mt.tivo_next_network_affiliate, mt.tms_next_network_affiliate) END 
 END),1,40) as next_network_affiliate, 
    substring((CASE WHEN 'vizio' != 'nielsen' AND mt.prev_nielsen_exclusive THEN NULL 
      WHEN (mt.prev_station_blacklist_clients IS NOT NULL 
            AND mt.prev_station_blacklist_clients != '||' 
            AND mt.prev_station_blacklist_clients NOT LIKE '%|vizio|%') OR mt.prev_station_blacklist_clients <=> '|ALL|' THEN NULL 
      ELSE mt.live END ),1,1) as live, 
    substring((mt.brand_name),1,256) as brand_name, 
    substring((mt.title),1,256) as title, 
    substring((mt.duration),1,5) as duration, 
    substring((mt.ip),1,32) as ip, 
    substring((mt.input_category),1,16) as input_category, 
    substring((mt.input_device),1,32) as input_device, 
    substring((CASE WHEN (mt.appb_clients IS NOT NULL 
            AND mt.appb_clients != '||' 
            AND mt.appb_clients NOT LIKE '%|vizio|%') OR mt.appb_clients <=> '|ALL|' THEN 'OBFUSCATED' 
      ELSE mt.app_service 
 END ),1,32) as app_service, 
    substring((mt.vizio_epg_channel_id),1,32) as vizio_epg_channel_id, 
    substring((mt.vizio_epg_program_id),1,32) as vizio_epg_program_id
    FROM (
        SELECT viz.*
        FROM prod.detection.viewing_commercials_cfe_golden AS viz
        WHERE CASE WHEN (viz.acrb_clients IS NOT NULL
            AND viz.acrb_clients != '||'
            AND viz.acrb_clients NOT LIKE '%|vizio|%') OR viz.acrb_clients <=> '|ALL|' THEN FALSE ELSE TRUE END
--     UNION
--         SELECT onn.* FROM prod.detection_onn.viewing_commercials_cfe_golden AS onn
--         WHERE CASE WHEN (onn.acrb_clients IS NOT NULL
--             AND onn.acrb_clients != '||'
--             AND onn.acrb_clients NOT LIKE '%|vizio|%') OR onn.acrb_clients <=> '|ALL|' THEN FALSE ELSE TRUE END
    ) AS mt
WHERE 1=1

In [0]:
%sql
CREATE OR REPLACE VIEW dev.mohit_gangwani.test_enhanced_kinetiq_2026070715_refined AS
SELECT /*+ BROADCAST(tp)
         */
    mt.tvid AS tvid, 
    mt.zipcode as zipcode, 
    mt.dma as dma, 
    mt.external_id as value, 
    mt.mt_start as mt_start, 
    mt.session_start as ts_start, 
    mt.session_end as ts_end, 
    CASE WHEN mt.prev_filter THEN NULL
         ELSE mt.tms_prev_episode_id END as prev_episode_id, 
    CASE WHEN mt.prev_filter THEN NULL
         ELSE COALESCE(mt.tms_prev_show_title, mt.tivo_prev_show_title) END as prev_title, 
    mt.prev_ts_start as prev_ts_start, 
    mt.prev_ts_end as prev_ts_end, 
    CASE WHEN mt.prev_filter THEN NULL
         ELSE COALESCE(mt.tms_prev_callsign, mt.tivo_prev_callsign) END as prev_channel_callsign, 
    CASE WHEN mt.prev_filter THEN NULL
         ELSE COALESCE(mt.tms_prev_network_affiliate, mt.tivo_prev_network_affiliate) END as prev_network_affiliate, 
    CASE WHEN mt.next_filter THEN NULL
         ELSE mt.tms_next_episode_id END as next_episode_id,
    CASE WHEN mt.next_filter THEN NULL 
         ELSE COALESCE(mt.tms_next_show_title, mt.tivo_next_show_title) END as next_title, 
    mt.next_ts_start as next_ts_start, 
    mt.next_ts_end as next_ts_end, 
    CASE WHEN mt.next_filter THEN NULL 
         ELSE COALESCE(mt.tms_next_callsign, mt.tivo_next_callsign) END as next_channel_callsign, 
    CASE WHEN mt.next_filter THEN NULL 
         ELSE COALESCE(mt.tms_next_network_affiliate, mt.tivo_next_network_affiliate) END as next_network_affiliate, 
    CASE WHEN mt.prev_filter THEN NULL 
         ELSE mt.live END as live, 
    mt.brand_name as brand_name, 
    mt.title as title, 
    mt.duration as duration, 
    mt.ip as ip, 
    mt.input_category as input_category, 
    mt.input_device as input_device, 
    CASE WHEN (mt.appb_clients IS NOT NULL 
            AND mt.appb_clients != '||' 
            AND mt.appb_clients NOT LIKE '%|vizio|%') OR mt.appb_clients <=> '|ALL|' THEN 'OBFUSCATED' 
      ELSE mt.app_service 
 END as app_service, 
    mt.vizio_epg_channel_id as vizio_epg_channel_id, 
    mt.vizio_epg_program_id as vizio_epg_program_id
FROM (
     SELECT viz.*,
     viz.next_nielsen_exclusive
     OR (NOT(reported_input_source <=> 'ANTENNA') AND COALESCE(next_station_blacklist_clients, '||') != '||') AS next_filter,
     viz.prev_nielsen_exclusive
     OR (NOT(reported_input_source <=> 'ANTENNA') AND COALESCE(prev_station_blacklist_clients, '||') != '||') AS prev_filter
     FROM prod.detection.viewing_commercials_cfe_golden AS viz
     WHERE (COALESCE(viz.acrb_clients, '||') = '||' OR viz.acrb_clients LIKE '%|vizio|%')
--     UNION ALL
--      SELECT onn.*,
--      onn.next_nielsen_exclusive
--      OR (NOT(onn.reported_input_source <=> 'ANTENNA') AND COALESCE(onn.next_station_blacklist_clients, '||') != '||') AS next_filter,
--      onn.prev_nielsen_exclusive
--      OR (NOT(onn.reported_input_source <=> 'ANTENNA') AND COALESCE(onn.prev_station_blacklist_clients, '||') != '||') AS prev_filter
--      FROM prod.detection_onn.viewing_commercials_cfe_golden AS onn
--      WHERE (COALESCE(onn.acrb_clients, '||') = '||' OR onn.acrb_clients LIKE '%|vizio|%')
) AS mt
GROUP BY ALL

## Performance Benchmarks

The benchmark section below uses the same 3-day filter for both views:

```sql
WHERE ts_start >= CURRENT_DATE - INTERVAL 3 DAY
  AND ts_start < CURRENT_DATE
```

Recommended order:

1. Run the two `EXPLAIN COST` cells first to compare the optimized logical plans
2. Run the existing benchmark cell
3. Run the refined benchmark cell
4. Open the query profile for each run and compare wall-clock time, remote bytes, task time, Photon time, and shuffle-heavy operators

The benchmark result values were identical for both views:

| Metric | Existing | Refined |
| --- | --- | --- |
| `COUNT(*)` | 3,666,393,110 | 3,666,393,110 |
| `COUNT(DISTINCT hash)` | 13,643,633 | 13,643,633 |
| `COUNT(DISTINCT value)` | 757,347 | 757,347 |

That confirms the refined view preserved the output while changing the execution profile.

In [0]:
%sql
-- EXPLAIN: Existing view — look for full scan (no partition filter at scan node)
EXPLAIN COST
SELECT COUNT(1) AS session_count
, COUNT(DISTINCT tvid) AS tv_count
, COUNT(DISTINCT value) AS dist_comm_count
FROM dev.mohit_gangwani.test_enhanced_kinetiq_2026070715_existing
WHERE ts_start >= CURRENT_DATE - INTERVAL 3 DAY
AND ts_start < CURRENT_DATE

In [0]:
%sql
-- EXPLAIN: Refined view — look for partition pruning on commercial_client at scan node
EXPLAIN COST
SELECT COUNT(1) AS session_count
, COUNT(DISTINCT tvid) AS tv_count
, COUNT(DISTINCT value) AS dist_comm_count
FROM dev.mohit_gangwani.test_enhanced_kinetiq_2026070715_refined
WHERE ts_start >= CURRENT_DATE - INTERVAL 3 DAY
AND ts_start < CURRENT_DATE

## EXPLAIN COST — Plan Comparison & Findings

### Executive Summary

The refined view in Cell 10 is materially better than the existing view in Cell 9 and preserves the same output on the benchmark window. The improvement is structural, not just cosmetic:

* `tv_populations` is deduplicated before the join
* `UNION DISTINCT` is replaced with `UNION ALL`
* repeated blacklist logic is collapsed into reusable `prev_filter` and `next_filter` booleans

The result is a smaller plan, less join fan-out, lower Photon CPU time, and a meaningfully faster benchmark query.

### What Changed in the Plan

| Area | Existing (Cell 12) | Refined (Cell 13) | Why it matters |
| --- | --- | --- | --- |
| `tv_populations` branch | 1.05M rows flow into the join | `GROUP BY fk_tvid` reduces that branch to about 258K rows | Removes duplicate-device fan-out before the main join |
| Source combination | `UNION` with an Aggregate-over-Union shape | direct `UNION ALL` node | Removes branch-level dedup work between Vizio and ONN |
| Projection logic | blacklist and Nielsen checks repeated in many CASE expressions | CASE expressions reference `prev_filter` and `next_filter` | Lowers repeated projection CPU work |
| Downstream size estimate | much larger join/project/aggregate estimates | about 77–78% smaller through those nodes | Less data flowing into the expensive global DISTINCT |

### Measured Benchmark Results

Benchmark query profiles used for the summary below:

* existing view: `5d6dbdf3-1c67-4116-a221-2b5ff818b920`
* refined view: `3f2212a0-e755-4384-ae17-c6ca457fb4c5`

| Metric | Existing | Refined | Improvement |
| --- | --- | --- | --- |
| `COUNT(*)` | 3,666,393,110 | 3,666,393,110 | same result |
| `COUNT(DISTINCT hash)` | 13,643,633 | 13,643,633 | same result |
| `COUNT(DISTINCT value)` | 757,347 | 757,347 | same result |
| Total duration | 1,221.6 sec | 682.9 sec | **44% faster** |
| Remote bytes read | 713.7 GiB | 285.5 GiB | **60% lower** |
| Total task time | 66.4h | 40.0h | **40% lower** |
| Photon CPU time | 114.5h | 38.1h | **67% lower** |
| Top shuffle time | 10.94h | 6.73h | **38% lower** |

### Interpretation

The most important signal is the drop in Photon CPU time from 114.5h to 38.1h. That aligns with the two biggest query-shape improvements:

* fewer rows entering the main join because `tv_populations` is deduplicated first
* less repeated expression work because the outer projection reuses `prev_filter` and `next_filter`

The refined view did read more files and partitions, but that did not hurt the final runtime. In this case that is consistent with `UNION ALL` allowing the two source-table scans to remain independent while still reducing the amount of expensive downstream work.

### Remaining Bottleneck

The outer `SELECT DISTINCT` is still the dominant cost center in both versions. Even after the refinements, shuffle and hash aggregate operators remain the top expensive stages in the profile. If more optimization is needed, the next step is to investigate whether the final DISTINCT can be narrowed or replaced with a surrogate-key dedup strategy.

> Cache note: the refined benchmark ran immediately after the existing benchmark and had a much higher cache-hit rate. That means the wall-clock gain is somewhat cache-assisted, but the lower task time and much lower Photon time show that the refined query is still structurally better.

In [0]:
%sql
-- Benchmark: Existing view
-- After this runs, open the Query Profile and note: files read, bytes read, duration
SELECT COUNT(1) AS session_count
, COUNT(DISTINCT tvid) AS tv_count
, COUNT(DISTINCT value) AS dist_comm_count
FROM dev.mohit_gangwani.test_enhanced_kinetiq_2026070715_existing
WHERE ts_start >= CURRENT_DATE - INTERVAL 3 DAY
AND ts_start < CURRENT_DATE

In [0]:
%sql
-- Benchmark: Refined view
-- After this runs, open the Query Profile and compare files read, bytes read, duration vs. existing view above
SELECT COUNT(1) AS session_count
, COUNT(DISTINCT tvid) AS tv_count
, COUNT(DISTINCT value) AS dist_comm_count
FROM dev.mohit_gangwani.test_enhanced_kinetiq_2026070715_refined
WHERE ts_start >= CURRENT_DATE - INTERVAL 3 DAY
AND ts_start < CURRENT_DATE

In [0]:
%sql
SELECT COUNT(1) AS session_count
, COUNT(DISTINCT tvid) AS tv_count
, COUNT(DISTINCT value) AS dist_comm_count
FROM prod.datalake_share.enhanced_kinetiq_springserve_commercial
WHERE ts_start >= CURRENT_DATE - INTERVAL 8 DAY
AND ts_start < CURRENT_DATE

In [0]:
%sql
ALTER VIEW cure_deltashare.vizio.enhanced_kinetiq_springserve_commercial AS
SELECT 
    mt.tvid AS tvid, 
    mt.zipcode as zipcode, 
    mt.dma as dma, 
    mt.external_id as value, 
    mt.mt_start as mt_start, 
    mt.session_start as ts_start, 
    mt.session_end as ts_end, 
    CASE WHEN mt.prev_filter THEN NULL
         ELSE mt.tms_prev_episode_id END as prev_episode_id, 
    CASE WHEN mt.prev_filter THEN NULL
         ELSE COALESCE(mt.tms_prev_show_title, mt.tivo_prev_show_title) END as prev_title, 
    mt.prev_ts_start as prev_ts_start, 
    mt.prev_ts_end as prev_ts_end, 
    CASE WHEN mt.prev_filter THEN NULL
         ELSE COALESCE(mt.tms_prev_callsign, mt.tivo_prev_callsign) END as prev_channel_callsign, 
    CASE WHEN mt.prev_filter THEN NULL
         ELSE COALESCE(mt.tms_prev_network_affiliate, mt.tivo_prev_network_affiliate) END as prev_network_affiliate, 
    CASE WHEN mt.next_filter THEN NULL
         ELSE mt.tms_next_episode_id END as next_episode_id,
    CASE WHEN mt.next_filter THEN NULL 
         ELSE COALESCE(mt.tms_next_show_title, mt.tivo_next_show_title) END as next_title, 
    mt.next_ts_start as next_ts_start, 
    mt.next_ts_end as next_ts_end, 
    CASE WHEN mt.next_filter THEN NULL 
         ELSE COALESCE(mt.tms_next_callsign, mt.tivo_next_callsign) END as next_channel_callsign, 
    CASE WHEN mt.next_filter THEN NULL 
         ELSE COALESCE(mt.tms_next_network_affiliate, mt.tivo_next_network_affiliate) END as next_network_affiliate, 
    CASE WHEN mt.prev_filter THEN NULL 
         ELSE mt.live END as live, 
    mt.brand_name as brand_name, 
    mt.title as title, 
    mt.duration as duration, 
    mt.ip as ip, 
    mt.input_category as input_category, 
    mt.input_device as input_device, 
    CASE WHEN (mt.appb_clients IS NOT NULL 
            AND mt.appb_clients != '||' 
            AND mt.appb_clients NOT LIKE '%|vizio|%') OR mt.appb_clients <=> '|ALL|' THEN 'OBFUSCATED' 
      ELSE mt.app_service 
 END as app_service, 
mt.vizio_epg_channel_id as vizio_epg_channel_id, 
mt.vizio_epg_program_id as vizio_epg_program_id,
mt.session_start_hour AS date_partition
FROM (
     SELECT viz.*,
     viz.next_nielsen_exclusive
     OR (NOT(reported_input_source <=> 'ANTENNA') AND COALESCE(next_station_blacklist_clients, '||') != '||') AS next_filter,
     viz.prev_nielsen_exclusive
     OR (NOT(reported_input_source <=> 'ANTENNA') AND COALESCE(prev_station_blacklist_clients, '||') != '||') AS prev_filter
     FROM prod.detection.viewing_commercials_cfe_golden AS viz
    UNION ALL
     SELECT onn.*,
     onn.next_nielsen_exclusive
     OR (NOT(onn.reported_input_source <=> 'ANTENNA') AND COALESCE(onn.next_station_blacklist_clients, '||') != '||') AS next_filter,
     onn.prev_nielsen_exclusive
     OR (NOT(onn.reported_input_source <=> 'ANTENNA') AND COALESCE(onn.prev_station_blacklist_clients, '||') != '||') AS prev_filter
     FROM prod.detection_onn.viewing_commercials_cfe_golden AS onn
) AS mt
WHERE (mt.acrb_clients <=> '|ALL|' AND '' <=> mt.app_service)
   OR COALESCE(mt.acrb_clients, '||') = '||'
   OR mt.acrb_clients LIKE '%|vizio|%'
GROUP BY ALL